# Motor cortex scANVI model

In [ ]:
# Data Download
# Data aquired from https://portal.brain-map.org/atlases-and-data/rnaseq/human-m1-10x

# Run on 20221020

! wget https://idk-etl-prod-download-bucket.s3.amazonaws.com/aibs_human_m1_10x/matrix.csv
! wget https://idk-etl-prod-download-bucket.s3.amazonaws.com/aibs_human_m1_10x/metadata.csv
! wget https://idk-etl-prod-download-bucket.s3.amazonaws.com/aibs_human_m1_10x/tsne.csv


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scarches.dataset.trvae.data_handling import remove_sparsity
import torch

import anndata
import scvi
import scanpy as sc
import wandb

In [2]:
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning import Trainer

In [3]:
sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

scvi.settings.seed = 94705


%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'

Global seed set to 94705


In [ ]:
torch.cuda.is_available()

In [5]:
import tensorflow as tf

In [ ]:
tf.test.is_gpu_available(
    cuda_only=False, min_cuda_compute_capability=None
)

In [ ]:
tf.config.list_physical_devices('GPU')

In [5]:
data_dir = '/Allen_Brain_Atlas/M1/'

In [6]:
adata = sc.read_csv(f'{data_dir}/matrix.csv')

In [7]:
adata.obs = pd.read_csv(f'{data_dir}/metadata.csv', index_col=0, header=0)

In [ ]:
adata.obs.keys()

In [ ]:
adata.obs[['cluster_label', 'class_label', 'subclass_label', 'cell_type_accession_label',
       'cell_type_alias_label', 'cell_type_designation_label', 'external_donor_name_label']]

In [ ]:
adata.obs.external_donor_name_label.value_counts()

In [11]:
reference_adata = adata

In [12]:
reference_adata = remove_sparsity(reference_adata)

In [13]:
if not reference_adata.raw:
    reference_adata.raw = reference_adata

In [14]:
# Fill the counts layer with the raw counts
reference_adata.layers['counts'] = reference_adata.raw.X if reference_adata.raw else reference_adata.X

In [ ]:
reference_adata.write_h5ad(f'{data_dir}/Allen_Brain_Atlas_M1_raw.h5ad', compression='gzip')

In [16]:
reference_adata.obs["labels_scanvi"] = reference_adata.obs["cluster_label"].values

In [17]:
sc.pp.normalize_total(reference_adata)
sc.pp.log1p(reference_adata)

In [ ]:
reference_adata.obs['external_donor_name_label'] # This should be a key specific to the dataset origin, create it if it doesn't exist

In [19]:
condition_key = 'external_donor_name_label'

In [ ]:
reference_adata.obs['cluster_label'] # This should be a key which contains the cell type information to be learned

In [21]:
cell_type_key = 'cluster_label'

In [22]:
sc.pp.highly_variable_genes(
    reference_adata,
    n_top_genes=2000,
    batch_key=condition_key,
    subset=True)

In [23]:
reference_adata.X = reference_adata.raw[:, reference_adata.var_names].X

In [24]:
scvi.model.SCVI.setup_anndata(reference_adata, 
                              batch_key=condition_key, # Use a batch key if we have batches
                              layer="counts")

In [ ]:
arches_params = dict(
    use_layer_norm="both",
    use_batch_norm="none",
    encode_covariates=True,
    dropout_rate=0.2,
    n_layers=2,
)

vae_ref = scvi.model.SCVI(
    reference_adata,
    **arches_params
)

project_name = f'scVI__Allen_Brain_Atlas_M1__{condition_key}' if condition_key else 'scVI__Allen_Brain_Atlas_M1'
scvi_wandb_logger = WandbLogger(project=project_name)
vae_ref.train(logger=scvi_wandb_logger)

In [ ]:
vae_ref_scan = scvi.model.SCANVI.from_scvi_model(
    vae_ref,
    unlabeled_category="Unknown",
    labels_key="labels_scanvi",
)

project_name = f'scANVI__Allen_Brain_Atlas_M1__{condition_key}' if condition_key else 'scANVI__Allen_Brain_Atlas_M1'
project_name += f'__{cell_type_key}'
wandb.finish()
scanvi_wandb_logger = WandbLogger(project=project_name)
logger_kwargs = {'logger': scanvi_wandb_logger}
vae_ref_scan.train(max_epochs=20, n_samples_per_label=100, **logger_kwargs)

In [ ]:
reference_adata.obsm["X_scANVI"] = vae_ref_scan.get_latent_representation()
sc.pp.neighbors(reference_adata, use_rep="X_scANVI")
sc.tl.leiden(reference_adata)
sc.tl.umap(reference_adata)

In [ ]:
sc.pl.umap(
    reference_adata,
    color=[condition_key, cell_type_key],
    frameon=False,
    ncols=1,
)

In [32]:
from pathlib import Path 

In [33]:
models_base_dir = Path('/scANVI_Models/')

In [34]:
model_dir_path = models_base_dir / f"/Allen_Brain_Atlas/M1/{cell_type_key}/"

In [35]:
! mkdir -p {model_dir_path}

In [36]:
vae_ref_scan.save(model_dir_path, overwrite=True)

In [37]:
reference_adata.write_h5ad(f'{model_dir_path}/Allen_Brain_Atlas_M1_post_train.h5ad', compression='gzip')

In [ ]:
print(model_dir_path)

In [ ]:
! ls -la {model_dir_path}